<a href="https://colab.research.google.com/github/NicolasHinke/python_sis_26_2/blob/main/aula_14/modelo_atividade_aula14.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Atividade Prática 01 Aula 14

API
Você foi encarregado de criar um script de monitoramento de dados recebidos via API. Seu objetivo é consumir dados em tempo real da API, garantindo segurança no uso da chave de acesso e resiliência contra falhas de conexão.
Escolha uma API, informe nos comentários qual é a API e a página de documentação.

Requisitos:  
*Crie scripts em Python que executem as etapas a seguir:

Etapa 1:
Configure a consulta à WeatherAPI utilizando a biblioteca requests. Armazene sua API_KEY de forma segura (usando getpass ou Secrets do Google Colab, sem deixá-la exposta no código) e defina uma lista com pelo menos 10 cidades brasileiras de sua escolha.

In [ ]:

import requests
from getpass import getpass

# API utilizada: WeatherAPI
# Documentação: https://www.weatherapi.com/docs/

API_KEY = getpass("Digite sua chave da WeatherAPI: ")

cidades = [
    "Curitiba",
    "São Paulo",
    "Rio de Janeiro",
    "Belo Horizonte",
    "Porto Alegre",
    "Florianópolis",
    "Salvador",
    "Brasília",
    "Recife",
    "Fortaleza"
]

URL = "https://api.weatherapi.com/v1/current.json"

respostas_http = []
resultados = []

print("Configuração concluída!")
print(f"Total de cidades: {len(cidades)}")


Etapa 2:
Crie um laço de repetição para consultar o endpoint para cada cidade da lista. Utilize timeout=10 na requisição e valide a resposta com raise_for_status() dentro de um bloco try/except, exibindo o código de status HTTP (status_code) e tratando possíveis erros (como cidade inválida ou falha de rede).

In [ ]:

for cidade in cidades:
    parametros = {
        "key": API_KEY,
        "q": cidade,
        "lang": "pt"
    }

    try:
        resposta = requests.get(
            URL,
            params=parametros,
            timeout=10
        )

        print(f"\nCidade: {cidade}")
        print(f"Status HTTP: {resposta.status_code}")

        resposta.raise_for_status()
        respostas_http.append(resposta)

    except requests.exceptions.HTTPError:
        print(f"Erro HTTP ao consultar {cidade}.")

    except requests.exceptions.Timeout:
        print(f"Tempo limite excedido para {cidade}.")

    except requests.exceptions.RequestException as erro:
        print(f"Falha de conexão em {cidade}: {erro}")

print("\nConsultas finalizadas!")
print(f"Respostas válidas: {len(respostas_http)}")



Etapa 3:
Converta as respostas válidas para dicionário Python com .json(), inspecione as chaves principais (keys()) e armazene todos os resultados em uma lista. Ao final, exiba no console o nome da cidade, a temperatura atual (temp_c) e a condição do tempo (condition -> text) de cada registro coletado.

In [ ]:

resultados = []

for resposta in respostas_http:
    try:
        dados = resposta.json()
        print("\nChaves principais:", dados.keys())

        resultados.append(dados)

        cidade = dados["location"]["name"]
        temperatura = dados["current"]["temp_c"]
        condicao = dados["current"]["condition"]["text"]

        print(f"Cidade: {cidade}")
        print(f"Temperatura: {temperatura} °C")
        print(f"Condição: {condicao}")

    except (ValueError, KeyError) as erro:
        print(f"Erro ao processar resposta: {erro}")

print("\nTotal de registros coletados:", len(resultados))


# Atividade Prática 02 Aula 14

Pandas
Com os dados coletados na Atividade 01,  gere uma análise consolidada e visual para tomada de decisão. Seu objetivo é transformar o JSON/XML em um DataFrame estruturado, tratar os dados e gerar um relatório visual e auditável.
Requisitos:  
*Crie scripts em Python que executem as etapas a seguir:


Etapa 1:
Converta a lista de dicionários da Atividade 01 em um DataFrame. Selecione apenas as colunas relevantes e renomeie-as para nomes claros em português.




In [ ]:

import pandas as pd

dados_tabela = []

for item in resultados:
    dados_tabela.append({
        "Cidade": item["location"]["name"],
        "Regiao": item["location"]["region"],
        "Pais": item["location"]["country"],
        "Data_Hora": item["location"]["localtime"],
        "Temperatura_C": item["current"]["temp_c"],
        "Umidade": item["current"]["humidity"],
        "Vento_kmh": item["current"]["wind_kph"],
        "Condicao": item["current"]["condition"]["text"]
    })

df = pd.DataFrame(dados_tabela)

print("DataFrame criado com sucesso!")
display(df)


Etapa 2:
Realize o tratamento e a tipagem dos dados: remova possíveis registros duplicados, converta colunas de data/hora com pd.to_datetime() e formato numérico com pd.to_numeric(). Em seguida, exiba um resumo estatístico.


In [ ]:

if not df.empty:
    df = df.drop_duplicates()

    df["Data_Hora"] = pd.to_datetime(
        df["Data_Hora"],
        errors="coerce"
    )

    colunas_numericas = [
        "Temperatura_C",
        "Umidade",
        "Vento_kmh"
    ]

    for coluna in colunas_numericas:
        df[coluna] = pd.to_numeric(
            df[coluna],
            errors="coerce"
        )

    print("Resumo estatístico:")
    display(df.describe())

    print("\nInformações do DataFrame:")
    df.info()

else:
    print("Nenhum dado foi coletado. Verifique sua chave da API.")


Etapa 3:
Gere um gráfico utilizando matplotlib comparando dados  que respondam a alguma pergunta, aplicando as boas práticas visuais. Por fim, exporte o DataFrame tratado para um arquivo .csv ou .xlsx.


In [ ]:

import matplotlib.pyplot as plt

if not df.empty:
    plt.figure(figsize=(12, 6))

    plt.bar(
        df["Cidade"],
        df["Temperatura_C"]
    )

    plt.title(
        "Temperatura Atual nas Cidades Brasileiras",
        fontsize=15,
        fontweight="bold"
    )

    plt.xlabel("Cidades")
    plt.ylabel("Temperatura (°C)")
    plt.xticks(rotation=45)
    plt.grid(axis="y", linestyle="--", alpha=0.4)

    plt.tight_layout()
    plt.show()

    df.to_csv(
        "relatorio_clima_aula14.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("Arquivo relatorio_clima_aula14.csv exportado com sucesso!")

else:
    print("Não foi possível gerar o gráfico ou exportar os dados.")
